DeepEval Agentic Metrics Evaluation

This project validates an AI agent workflow using DeepEval agentic metrics.

An agent usually does:
user request
→ plans or decides steps
→ selects tools
→ calls tools
→ uses tool results
→ gives final answer

Agentic metrics check:
- Did the agent complete the task?
- Did the agent choose the correct tool?
- Did the agent pass correct arguments?
- Did the agent follow the plan?
- Did the agent avoid unnecessary steps?

This project evaluates AI agent behavior using DeepEval agentic metrics.

In [1]:
!pip install deepeval groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.5/601.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [2]:
import os
from google.colab import userdata
from groq import Groq

from deepeval.models import DeepEvalBaseLLM

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


class GroqDeepEvalModel(DeepEvalBaseLLM):

    def __init__(self, model_name="openai/gpt-oss-20b"):
        self.model_name = model_name
        self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def load_model(self):
        return self.client

    def generate(self, prompt: str, **kwargs) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": "Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content

    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)

    def get_model_name(self):
        return self.model_name


groq_model = GroqDeepEvalModel()

print("Groq evaluator connected.")

Groq evaluator connected.


In [4]:
from google.colab import files

uploaded = files.upload()

Saving agent.py to agent.py


In [5]:
from agent import support_agent

print("Real agent imported successfully.")

Real agent imported successfully.


In [11]:
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.test_case import ToolCall

agent_goldens = []

for _, row in agent_df.iterrows():
    agent_goldens.append(
        Golden(
            input=row["user_task"],
            expected_output=row["expected_outcome"],
            expected_tools=[
                ToolCall(name=str(row["expected_tools"]).strip())
            ],
        )
    )

agent_dataset = EvaluationDataset(goldens=agent_goldens)

print("DeepEval dataset created.")
print("Total test cases:", len(agent_goldens))

DeepEval dataset created.
Total test cases: 35


In [12]:
from deepeval.metrics import ToolCorrectnessMetric

tool_correctness_metric = ToolCorrectnessMetric()

print("Tool Correctness metric created.")

DeepEvalError: OpenAI API key is not configured. Set OPENAI_API_KEY in your environment or pass `api_key` to OpenAIModel(...).

In [6]:
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.evaluate import ErrorConfig, AsyncConfig
from deepeval.tracing import observe, update_current_trace
from deepeval.metrics import TaskCompletionMetric, StepEfficiencyMetric, ToolCorrectnessMetric, ArgumentCorrectnessMetric, PlanQualityMetric, PlanAdherenceMetric

### TaskCompletionMetric

TaskCompletionMetric checks whether an AI agent successfully completed the user’s task.

It focuses on the final outcome of the agent’s work.

If the agent’s response fully satisfies the user request, it passes.

If the agent gives only partial help, unclear help, or does not complete the requested task, it fails.

In [7]:
task_completion_metric = TaskCompletionMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Task Completion metric created.")

Task Completion metric created.


In [9]:
import pandas as pd

agent_df = pd.read_csv("agentic_metrics_dataset.csv")

print("Agentic metrics dataset loaded.")
print("Total test cases:", len(agent_df))

display(agent_df.head())

Agentic metrics dataset loaded.
Total test cases: 35


,test_id,user_task,expected_outcome,expected_plan,expected_tools,expected_arguments,actual_output,actual_tools,actual_arguments
0,AGENT_TC_001,I placed order ORD1001 yesterday. How can I tr...,Provide clear tracking guidance for ORD1001 us...,Identify order tracking intent; extract ORD100...,track_order_tool,"{""order_id"": ""ORD1001""}",Order ORD1001 can be tracked using the trackin...,track_order_tool,"{""order_id"": ""ORD1001""}"
1,AGENT_TC_002,The product is unused and I bought it 15 days ...,Confirm refund eligibility for an unused produ...,Identify refund eligibility intent; check prod...,refund_policy_tool,"{""product_condition"": ""unused"", ""days_since_pu...",An unused product in original condition can be...,refund_policy_tool,"{""product_condition"": ""unused"", ""days_since_pu..."
2,AGENT_TC_003,My order ORD1003 has not shipped yet. I need t...,Explain that the address can be changed before...,Identify address-change intent; extract ORD100...,change_address_tool,"{""order_id"": ""ORD1003"", ""shipment_status"": ""no...",Delivery address for order ORD1003 can be chan...,change_address_tool,"{""order_id"": ""ORD1003"", ""shipment_status"": ""no..."
3,AGENT_TC_004,"I already paid for ORD1004, but it has not bee...",Confirm cancellation is possible before shippi...,Identify cancellation intent; extract order/pa...,cancel_order_tool,"{""order_id"": ""ORD1004"", ""payment_status"": ""pai...",Order ORD1004 can be cancelled before shipping...,cancel_order_tool,"{""order_id"": ""ORD1004"", ""payment_status"": ""pai..."
4,AGENT_TC_005,"Money was deducted from my account, but I did ...",Ask for transaction details and guide the user...,Identify payment failure intent; call payment_...,payment_issue_tool,"{""issue_type"": ""payment_deducted_order_not_cre...",If payment was deducted but the order was not ...,payment_issue_tool,"{""issue_type"": ""payment_deducted_order_not_cre..."


This approach is used when QA does not have access to the live agent source code or tracing callbacks. In that case, actual_tools and expected_tools can be compared from logs/CSV.

If agent source code is available, ToolCorrectnessMetric can be implemented in DeepEval’s tracing style using @observe, Golden(expected_tools=...), EvaluationDataset, and evals_iterator(), where actual tool calls are captured automatically from the agent execution.

In [10]:
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
import time

tool_correctness_results = []

for row_index, row in agent_df.iterrows():
    task_completion_test_case = LLMTestCase(
        input=row["user_task"],
        actual_output=row["actual_output"]
    )

    print("Running:", row["test_id"])

    evaluate(
        test_cases=[task_completion_test_case],
        metrics=[task_completion_metric],
        async_config=AsyncConfig(run_async=False),
        error_config=ErrorConfig(ignore_errors=True)
    )

    actual_tool = str(row["actual_tools"]).strip()
    expected_tool = str(row["expected_tools"]).strip()

    tool_correctness_status = "PASS" if actual_tool == expected_tool else "FAIL"

    tool_correctness_results.append({
        "test_id": row["test_id"],
        "user_task": row["user_task"],
        "expected_tool": expected_tool,
        "actual_tool": actual_tool,
        "tool_correctness": tool_correctness_status
    })

    print("Expected tool:", expected_tool)
    print("Actual tool:", actual_tool)
    print("Tool Correctness:", tool_correctness_status)
    print("Completed:", row["test_id"])
    print("-" * 80)

    time.sleep(5)

tool_correctness_df = pd.DataFrame(tool_correctness_results)

display(tool_correctness_df)

print("Task Completion + Tool Correctness evaluation completed.")

Running: AGENT_TC_001


✨ You're running DeepEval's latest Task Completion Metric! (using openai/gpt-oss-20b, strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            I placed order ORD1001 yesterday. How can I track where it is now?                     │
│  │     Actual Output:    Order ORD1001 can be tracked using the tracking link sent to your registered email     │
│  │                       or mobile number.                                                                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric          ┃ Score ┃ Threshold ┃ Reason                                                     │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Task Completion │ 0.20  │ 0.60      │ The response only tells the user how to track the order    │
│              │                 │       │           │ via a link, but it does not provide the actual status of   │
│              │                 │       │           │ order ORD1001, which is the core of the requested task.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                    ┃ Average Score          ┃ Pass Rate                                    ┃ Total      │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━ │
│  Task Completion           │ 0.20                   │ 0.00% | passed=0 | failed=1                  │ 1          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=395835;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

...

⚠ WARNING: No hyperparameters logged.
» ]8;id=749247;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.94s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Expected tool: review_tool
Actual tool: payment_issue_tool
Tool Correctness: FAIL
Completed: AGENT_TC_035
--------------------------------------------------------------------------------


,test_id,user_task,expected_tool,actual_tool,tool_correctness
0,AGENT_TC_001,I placed order ORD1001 yesterday. How can I tr...,track_order_tool,track_order_tool,PASS
1,AGENT_TC_002,The product is unused and I bought it 15 days ...,refund_policy_tool,refund_policy_tool,PASS
2,AGENT_TC_003,My order ORD1003 has not shipped yet. I need t...,change_address_tool,change_address_tool,PASS
3,AGENT_TC_004,"I already paid for ORD1004, but it has not bee...",cancel_order_tool,cancel_order_tool,PASS
4,AGENT_TC_005,"Money was deducted from my account, but I did ...",payment_issue_tool,payment_issue_tool,PASS
5,AGENT_TC_006,I need the invoice for order ORD1006. Where ca...,get_invoice_tool,get_invoice_tool,PASS
6,AGENT_TC_007,When will my order ORD1007 reach Bengaluru?,delivery_estimate_tool,delivery_estimate_tool,PASS
7,AGENT_TC_008,"Before placing an order, I want to know what d...",delivery_options_tool,delivery_options_tool,PASS
8,AGENT_TC_009,My order is delayed and I want to talk to a re...,human_handoff_tool,human_handoff_tool,PASS
9,AGENT_TC_010,The product I received is damaged. How do I ra...,complaint_tool,complaint_tool,PASS


Task Completion + Tool Correctness evaluation completed.


StepEfficiencyMetric

StepEfficiencyMetric checks whether an AI agent completed the task using useful and necessary steps.

It focuses on the agent’s execution path.

If the agent uses direct and relevant steps, it passes.

If the agent uses unnecessary, repeated, or unrelated steps, it fails.

In [ ]:
step_efficiency_metric = StepEfficiencyMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Step Efficiency metric created.")

Step Efficiency metric created.


In [ ]:
for golden in agent_dataset.evals_iterator(
    metrics=[step_efficiency_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Agentic Step Efficiency evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                    ┃ Average Score         ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Step Efficiency           │ 1.00                  │ 100.00% | passed=3 | failed=0                  │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=338782;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 36.59s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Agentic Step Efficiency evaluation completed.


ToolCorrectnessMetric

ToolCorrectnessMetric checks whether an AI agent selected the correct tool for the given task.

It compares the tools actually used by the agent with the tools expected for that task.

If the agent calls the correct tool, it passes.

If the agent calls the wrong tool, misses a required tool, or uses an unnecessary tool, it fails.

In [ ]:
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval import evaluate

In [ ]:
tool_correctness_metric = ToolCorrectnessMetric(
    threshold=0.6
)

print("Tool Correctness metric created.")
# ToolCorrectnessMetric can compare expected tool and actual tool directly.

Tool Correctness metric created.


✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Tool Correctness           │ 1.00                  │ 100.00% | passed=3 | failed=0                 │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=970323;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.15s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Tool Correctness evaluation completed.


ArgumentCorrectnessMetric

ArgumentCorrectnessMetric checks whether an AI agent passed the correct arguments or input values into the selected tool.

It is used after checking tool correctness.

If the agent selects the correct tool but passes wrong, missing, or incomplete arguments, this metric can fail.

Example:
If the task is “Track order ORD123”, the agent should call the tracking tool with order_id = "ORD123".

If the agent calls the tracking tool without the order ID, the tool choice is correct, but the argument is wrong.

In [ ]:
argument_correctness_metric = ArgumentCorrectnessMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Argument Correctness metric created.")

Argument Correctness metric created.


In [ ]:
argument_test_cases = [
    LLMTestCase(
        input="Track order ORD123.",
        actual_output="Order ORD123 can be tracked using the tracking link.",
        tools_called=[
            ToolCall(
                name="track_order_tool",
                input_parameters={"order_id": "ORD123"}
            )
        ]
    ),
    LLMTestCase(
        input="Check refund policy for order ORD456.",
        actual_output="Order ORD456 is eligible for refund if it is within 15 days and unused.",
        tools_called=[
            ToolCall(
                name="refund_policy_tool",
                input_parameters={"order_id": "ORD456"}
            )
        ]
    ),
    LLMTestCase(
        input="Change delivery address for order ORD789.",
        actual_output="Delivery address for order ORD789 can be changed before shipment.",
        tools_called=[
            ToolCall(
                name="change_address_tool",
                input_parameters={"order_id": "ORD789"}
            )
        ]
    )
]

evaluate(
    test_cases=argument_test_cases,
    metrics=[argument_correctness_metric]
)

print("Argument Correctness evaluation completed.")

✨ You're running DeepEval's latest Argument Correctness Metric! (using openai/gpt-oss-20b, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:            Change delivery address for order ORD789.                                              │
│  │     Actual Output:    Delivery address for order ORD789 can be changed before shipment.                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Argument Correctness │ 0.00  │ 0.60      │ The score is 0.00 because the input only provides     │
│              │                      │       │           │ the order ID but does not include the new delivery    │
│              │                      │       │           │ address required to change the address.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                          ┃ Average Score        ┃ Pass Rate                                 ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Argument Correctness            │ 0.67                 │ 66.67% | passed=2 | failed=1              │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=98596;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.38s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Argument Correctness evaluation completed.


PlanQualityMetric

PlanQualityMetric checks whether an AI agent created a good plan before doing the task.

It focuses on the quality of the planned steps.

A good plan should be clear, logical, complete, and useful for completing the user’s task.

If the plan is missing important steps, has unnecessary steps, or is not useful for the task, this metric can fail.

In [ ]:
plan_quality_metric = PlanQualityMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Plan Quality metric created.")

Plan Quality metric created.


In [ ]:
@observe()
def planned_customer_support_agent(user_task):
    plan = [
        "Identify the customer support request.",
        "Select the correct support tool.",
        "Use the tool result to answer the customer."
    ]

    if "track" in user_task.lower():
        answer = track_order_tool()

    elif "refund" in user_task.lower():
        answer = refund_policy_tool()

    elif "delivery address" in user_task.lower() or "address" in user_task.lower():
        answer = change_address_tool()

    else:
        answer = "I could not identify the correct support action."

    update_current_trace(
        input=user_task,
        output=answer,
        metadata={
            "plan": plan
        }
    )

    return answer

In [ ]:
for golden in agent_dataset.evals_iterator(
    metrics=[plan_quality_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = planned_customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Plan Quality evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                ┃ Average Score           ┃ Pass Rate                                       ┃ Total      │
│ ━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━ │
│  Plan Quality          │ 1.00                    │ 100.00% | passed=3 | failed=0                   │ 3          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=677934;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.66s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Plan Quality evaluation completed.


PlanAdherenceMetric

PlanAdherenceMetric checks whether an AI agent followed the plan it created.

It compares the planned steps with the actual steps taken by the agent.

If the agent follows the planned steps properly, it passes.

If the agent skips planned steps, does different steps, or goes away from the plan, this metric can fail.

In [ ]:
plan_adherence_metric = PlanAdherenceMetric(
    threshold=0.6,
    model=groq_model,
    include_reason=True
)

print("Plan Adherence metric created.")

Plan Adherence metric created.


In [ ]:
for golden in agent_dataset.evals_iterator(
    metrics=[plan_adherence_metric],
    error_config=ErrorConfig(ignore_errors=True),
    async_config=AsyncConfig(run_async=False)
):
    answer = planned_customer_support_agent(golden.input)

    print("Task:", golden.input)
    print("Agent answer:", answer)
    print("-" * 80)

print("Plan Adherence evaluation completed.")

Output()

Task: Help the customer track their order.

Agent answer: Order can be tracked using the tracking link sent to the registered email or mobile number.

--------------------------------------------------------------------------------

Task: Help the customer understand the refund policy.

...

--------------------------------------------------------------------------------

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                   ┃ Average Score          ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Plan Adherence           │ 1.00                   │ 100.00% | passed=3 | failed=0                  │ 3         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=249628;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.81s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Plan Adherence evaluation completed.
